# Updated EDA: Cascading Flow Analysis with CFI Churn & CFI Rate
**Two-Period Comparison (2011 & 2021) — All Cascade Metrics**

### Metrics Compared
| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Net Cascade** | `Inflow_Wealthier − Outflow_Poorer` | Directional balance: positive = net gentrification pressure |
| **CFI Churn** | `Inflow_Wealthier + Outflow_Poorer` | Total cascade intensity — captures simultaneous displacement regardless of direction |
| **CFI Rate** | `(Inflow_W × Outflow_P) / Total_Migration` | Normalised interaction term — high only when *both* flows co-occur relative to turnover |
| **% Inflow Wealthier** | `Inflow_Wealthier / Total_Inflow × 100` | Share of all arrivals coming from wealthier areas |

### Data Source
`msoa_cascade_features_20260515.csv` — 983 London MSOAs, generated by `eda_update_cascade_features.ipynb`.

This analysis was using `IMD_score` for IMD related analysis, was trying to replacee with rank analysis metrics. I replaced all `IMD_change` with `IMD_Pctile_Change`, but just ran the correlation between it and all cascade features, and finding `CFI_rate` became significant with `IMD_Pctile_Change`, where it was insignificant with `IMD_change`(scores).

---

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pyprojroot import here

# Set visualization theme
sns.set_theme(style='whitegrid', font_scale=1.1)

# Define directories using pyprojroot
ROOT = here()
OUTPUT_DIR = ROOT / 'outputs'

# Load the pre-computed cascade features using the path object
data_path = OUTPUT_DIR / 'msoa_cascade_features_20260515.csv'
df = pd.read_csv(data_path)

# Compute deltas (2021 − 2011)
df['Delta_Net_Cascade'] = df['Net_Cascade_21'] - df['Net_Cascade_11']
df['Delta_CFI_Churn'] = df['CFI_Churn_21'] - df['CFI_Churn_11']
df['Delta_CFI_Rate'] = df['CFI_Rate_21'] - df['CFI_Rate_11']
df['Delta_Pct_Inflow_Wealthier'] = df['Pct_Inflow_Wealthier_21'] - df['Pct_Inflow_Wealthier_11']

# Display dataset summary
print(f'MSOAs: {len(df)}')
print(f'Boroughs: {df["ladnm"].nunique()}')

display(df.head())

---
## 2. Descriptive Statistics: Four Cascade Metrics

In [ ]:
cascade_cols = ['Net_Cascade', 'CFI_Churn', 'CFI_Rate', 'Pct_Inflow_Wealthier']
for base in cascade_cols:
    print(f'\n=== {base} ===')
    print(df[[f'{base}_11', f'{base}_21']].describe().round(2).to_string())

---
## 3. Distribution Comparison: 2011 vs 2021

Overlay histograms for each metric to assess whether cascade dynamics shifted over the decade.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = [
    ('Net_Cascade', 'Net Cascade (Inflow_W − Outflow_P)'),
    ('CFI_Churn', 'CFI Churn (Inflow_W + Outflow_P)'),
    ('CFI_Rate', 'CFI Rate (Inflow_W × Outflow_P / Total_Mig)'),
    ('Pct_Inflow_Wealthier', '% Inflow from Wealthier Areas'),
]

for ax, (base, title) in zip(axes.flat, metrics):
    c11, c21 = f'{base}_11', f'{base}_21'
    ax.hist(df[c11], bins=40, alpha=0.5, color='#4575b4', label='2011', edgecolor='white')
    ax.hist(df[c21], bins=40, alpha=0.5, color='#d73027', label='2021', edgecolor='white')
    ax.axvline(df[c11].median(), color='#4575b4', ls='--', lw=1.5, label=f'2011 median: {df[c11].median():.0f}')
    ax.axvline(df[c21].median(), color='#d73027', ls='--', lw=1.5, label=f'2021 median: {df[c21].median():.0f}')
    ax.set_xlabel(title)
    ax.set_ylabel('Number of MSOAs')
    ax.legend(fontsize=9)

fig.suptitle('Distribution of Cascade Metrics: 2011 vs 2021 (N=983 London MSOAs)', fontsize=14, y=1.01)
plt.tight_layout()
save_path = OUTPUT_DIR / 'fig8_metric_distributions.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

##### Interpretation
- **CFI Churn** and **Net Cascade** both shifted leftward (lower) from 2011 to 2021, suggesting overall migration volumes fell — likely a COVID-19 effect on the 2021 Census.
- The *shape* of the distributions matters more than the absolute level: compare which deciles were most affected.

---
## 4. Correlation Heatmap: All Metrics & IMD Change

This reveals how the four cascade formulations relate to each other and to the external validation variable (IMD Change).

In [ ]:
corr_cols = [
    'CFI_Churn_11','CFI_Rate_11','Net_Cascade_11','Pct_Inflow_Wealthier_11',
    'CFI_Churn_21','CFI_Rate_21','Net_Cascade_21','Pct_Inflow_Wealthier_21',
    'Delta_CFI_Churn','Delta_CFI_Rate','Delta_Net_Cascade','Delta_Pct_Inflow_Wealthier',
    'IMD_Pctile_Change', 'IMD_Score_Change'
]

fig, ax = plt.subplots(figsize=(14, 11))
cmat = df[corr_cols].corr()
sns.heatmap(cmat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in corr_cols],
            yticklabels=[c.replace('_', '\n') for c in corr_cols])
ax.set_title('Correlation Matrix: All Cascade Metrics & IMD Change', fontsize=13)
plt.tight_layout()
save_path = OUTPUT_DIR / 'fig9_correlation_heatmap.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

##### Key Observations
- **Net Cascade** has the strongest correlation with IMD Change (r ≈ −0.42), followed by **% Inflow Wealthier** (r ≈ −0.38) and **CFI Churn** (r ≈ −0.25 to −0.30).
- **CFI Rate** has essentially *no* correlation with IMD Change (r ≈ −0.05, not significant). This is an important finding — see Section 5 for discussion.
- **CFI Churn** and **CFI Rate** are strongly correlated with each other (r ≈ 0.74), but diverge in their relationship with deprivation change.

---
## 5. Validation: All Metrics vs IMD Change (Pearson r)

The key question: **which metric best predicts actual deprivation change?**

In [ ]:
print('=== Pearson Correlations with IMD Rank Change (2010→2019) ===')
print('  (negative r = metric rises where deprivation fell = expected gentrification signal)\n')

results = []
for col, label in [
    ('Net_Cascade_11', 'Net Cascade 2011'),
    ('Net_Cascade_21', 'Net Cascade 2021'),
    ('CFI_Churn_11', 'CFI Churn 2011'),
    ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'),
    ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Pct_Inflow_Wealthier_11', '% Inflow Wealthier 2011'),
    ('Pct_Inflow_Wealthier_21', '% Inflow Wealthier 2021'),
    ('Delta_Net_Cascade', 'Δ Net Cascade'),
    ('Delta_CFI_Churn', 'Δ CFI Churn'),
    ('Delta_CFI_Rate', 'Δ CFI Rate'),
    ('Delta_Pct_Inflow_Wealthier', 'Δ % Inflow Wealthier'),
]:
    valid = df[[col, 'IMD_Pctile_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Pctile_Change'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    results.append({'Metric': label, 'r': r, 'p': p, 'sig': sig})
    print(f'  {label:35s}  r = {r:+.3f}  p = {p:.2e}  {sig}')

val_df = pd.DataFrame(results)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d73027' if r < 0 else '#4575b4' for r in val_df['r']]
ax.barh(val_df['Metric'], val_df['r'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson r with IMD Change (2010→2019)')
ax.set_title('Validation: Cascade Metrics vs Deprivation Change\n(negative r = expected gentrification signal)')
for i, row in val_df.iterrows():
    ax.text(row['r'] + 0.01 * np.sign(row['r']), i,
            f"{row['r']:.3f} {row['sig']}", va='center', fontsize=9)
plt.tight_layout()

save_path = OUTPUT_DIR / 'fig10_validation_bar.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

##### Discussion: CFI measures

**CFI Rate** (r ≈ −0.05, *not significant*) does not predict deprivation change, despite being the most theoretically appealing normalised metric. This likely happens because:

1. **The interaction term creates a U-shape problem**: `(Inflow_W × Outflow_P) / Total_Migration` peaks for MSOAs in **middle deciles** (deciles 3–6) where both upward and downward flows are large. But these are not necessarily the MSOAs experiencing gentrification — they may simply be high-turnover transitional areas.

2. **Division by Total_Migration cancels out the signal**: Normalising by total migration removes the *volume* information that carries the gentrification signal. A wealthy area with small cascade flows and a deprived area with large cascade flows could have the same CFI Rate if their migration volumes scale proportionally.

3. **Decile 1 and 10 have zero CFI Rate by construction**: The most and least deprived deciles have no "wealthier inflow" or "poorer outflow" respectively, so their rates are mechanically zero regardless of actual dynamics.

**CFI Churn** (r ≈ −0.25 to −0.30, ***p < 0.001***) performs better because the additive formulation preserves volume information, but it still underperforms **Net Cascade** because it does not distinguish the *direction* of the cascade. An MSOA with 500 wealthier arrivals + 100 displaced residents has the same CFI Churn (600) as one with 100 arrivals + 500 displaced — but their gentrification dynamics are very different.

**Recommendation**: Net Cascade (or % Inflow Wealthier) remains the strongest single predictor. CFI Churn adds value as a measure of cascade *intensity* independent of direction, useful for the bivariate analysis in Section 8.

##### **Discussion: Why CFI Rate Fails Validation**

- Key Underlying Reason: The multiplicative term is only high when both components are simultaneously large.
    - An MSOA could be gentrifying strongly through heavy wealthy inflows alone (high Net_Cascade, high Churn) but if outflow to poorer areas is modest, the product stays low. 
    - The division by Total_Migration then strips out the population-size effect that drives the correlation in the other metrics. Larger MSOAs tend to have both higher absolute flows and more extreme IMD changes, so removing that shared scaling factor removes a major source of linear association.
- Gentrification signal in London is driven by the scale and directionality of flows, not by a normalised interaction between inflows and outflows

- CFI_Rate may be more useful for comparing areas of very different sizes (e.g., cross-city comparisons) where the raw volume metrics would be confounded by population, but within London's relatively comparable MSOAs, the simpler formulations perform better.

---
## 6. Cascade Metrics by Wealth Decile: 2011 vs 2021

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
x = np.arange(1, 11)
width = 0.35

for ax, base, title in [
    (axes[0,0], 'CFI_Churn', 'CFI Churn (Additive)'),
    (axes[0,1], 'CFI_Rate', 'CFI Rate (Normalised)'),
    (axes[1,0], 'Net_Cascade', 'Net Cascade (Directional)'),
    (axes[1,1], 'Pct_Inflow_Wealthier', '% Inflow from Wealthier'),
]:
    means = df.groupby('Wealth_Decile').agg(
        y11=(f'{base}_11', 'mean'), y21=(f'{base}_21', 'mean'))
    ax.bar(x - width/2, means['y11'], width, label='2011', color='#4575b4', edgecolor='white')
    ax.bar(x + width/2, means['y21'], width, label='2021', color='#d73027', edgecolor='white')
    ax.set_xlabel('Wealth Decile (1=most deprived, 10=wealthiest)')
    ax.set_ylabel(f'Mean {title}')
    ax.set_title(f'{title} by Decile: 2011 vs 2021')
    ax.set_xticks(x)
    ax.legend()
    if 'Net' in base:
        ax.axhline(0, color='black', lw=0.8)

plt.tight_layout()

save_path = OUTPUT_DIR / 'fig11_metrics_by_decile.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Observations
- **CFI Churn** has an inverted-U shape across deciles: highest in deciles 2–5, lowest at the extremes (1 and 10). This makes sense — cascade *churn* requires both inflow from above and outflow below, which is impossible at the decile endpoints.
- **CFI Rate** shows a similar inverted-U (peaking at deciles 4–5) but flatter, because normalisation dampens volume effects.
- **Net Cascade** shows a strong monotonic gradient: positive (net gentrification) in deprived deciles, negative in wealthy deciles. This is the expected cascade pattern. Spearman ρ ≈ −0.99 (p < 0.001).
- The 2021 values are generally lower than 2011 across all metrics, consistent with reduced migration during COVID-19.

---
## 7. Decile-Level Summary Table

In [ ]:
decile_full = df.groupby('Wealth_Decile').agg(
    N=('msoa11cd','count'),
    CFI_Churn_11=('CFI_Churn_11','mean'),
    CFI_Churn_21=('CFI_Churn_21','mean'),
    CFI_Rate_11=('CFI_Rate_11','mean'),
    CFI_Rate_21=('CFI_Rate_21','mean'),
    Net_Cascade_11=('Net_Cascade_11','mean'),
    Net_Cascade_21=('Net_Cascade_21','mean'),
    Pct_Inflow_W_11=('Pct_Inflow_Wealthier_11','mean'),
    Pct_Inflow_W_21=('Pct_Inflow_Wealthier_21','mean'),
    IMD_Rank_Change=('IMD_Pctile_Change','mean'),
).round(1)
display(decile_full)

In [ ]:
# Spearman monotonicity tests
print('=== Spearman Rank Correlations (Decile means vs Decile rank) ===\n')
for col, label in [
    ('CFI_Churn_11', 'CFI Churn 2011'), ('CFI_Churn_21', 'CFI Churn 2021'),
    ('CFI_Rate_11', 'CFI Rate 2011'), ('CFI_Rate_21', 'CFI Rate 2021'),
    ('Net_Cascade_11', 'Net Cascade 2011'), ('Net_Cascade_21', 'Net Cascade 2021'),
]:
    means = df.groupby('Wealth_Decile')[col].mean()
    rho, p = stats.spearmanr(means.index, means.values)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  Decile vs {label:25s}: ρ = {rho:+.3f}, p = {p:.4f} {sig}')

##### Key Result
- **Net Cascade** shows near-perfect monotonic decline across deciles (ρ = −0.99***) — a textbook cascade gradient.
- **CFI Churn** is weakly monotonic (ρ ≈ −0.6, borderline significant) — the inverted-U dilutes the gradient.
- **CFI Rate** is *not* monotonically related to wealth decile (ρ = −0.24, ns) — it peaks mid-distribution.

---
## 8. Bivariate Analysis: Gentrification Pressure vs Displacement Yield

This is where CFI Churn adds analytical value. By decomposing the additive index back into its two components,
we can map MSOAs on a "Pressure × Displacement" matrix — separating areas that *receive* gentrifiers from areas
that *export* displaced residents.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    sc = ax.scatter(
        df[f'Inflow_Wealthier{suffix}'], df[f'Outflow_Poorer{suffix}'],
        c=df['IMD_Pctile_Change'], cmap='RdBu', s=20, alpha=0.6,
        edgecolors='grey', linewidth=0.3, vmin=-20, vmax=10)
    ax.set_xlabel('Inflow from Wealthier Areas\n(Gentrification Pressure)')
    ax.set_ylabel('Outflow to More Deprived Areas\n(Displacement Yield)')
    ax.set_title(f'{year}: Gentrification Pressure vs Displacement Yield')
    lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='Balance line')
    ax.legend(loc='lower right')

cbar = fig.colorbar(sc, ax=axes, shrink=0.8)
cbar.set_label('IMD Change (2010→2019)\n(blue = became less deprived)')

save_path = OUTPUT_DIR / 'fig12_bivariate_pressure_displacement.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Interpretation
- MSOAs **above** the balance line have more displacement outflow than gentrification inflow → net displacement exporters.
- MSOAs **below** the line have more gentrification inflow → net gentrification receivers.
- The colour gradient confirms the pattern: blue points (large deprivation decline) cluster in the upper-right — high *simultaneous* pressure and displacement, i.e. high CFI Churn.

---
## 9. CFI Churn vs CFI Rate: Understanding the Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

for ax, suffix, year in [(axes[0], '_11', '2011'), (axes[1], '_21', '2021')]:
    sc = ax.scatter(
        df[f'CFI_Churn{suffix}'], df[f'CFI_Rate{suffix}'],
        c=df['Wealth_Decile'], cmap='RdYlGn', s=20, alpha=0.6,
        edgecolors='grey', linewidth=0.3)
    ax.set_xlabel('CFI Churn (Additive)')
    ax.set_ylabel('CFI Rate (Normalised)')
    ax.set_title(f'{year}: CFI Churn vs CFI Rate')
    r, p = stats.pearsonr(df[f'CFI_Churn{suffix}'], df[f'CFI_Rate{suffix}'])
    ax.text(0.05, 0.95, f'r = {r:.3f}', transform=ax.transAxes, fontsize=11,
            va='top', bbox=dict(boxstyle='round', fc='white', alpha=0.8))

cbar = fig.colorbar(sc, ax=axes, shrink=0.8)
cbar.set_label('Wealth Decile (10=wealthiest)')

save_path = OUTPUT_DIR / 'fig13_churn_vs_rate.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Observation
- The two metrics are correlated (r ≈ 0.74) but not identical. The scatter fans out because CFI Rate penalises high-turnover MSOAs — areas with large Total_Migration get their Rate pulled down even if cascade flows are high.
- Green points (wealthy deciles) cluster at the origin (low churn, low rate) while red/yellow points (deprived deciles) spread across the range.

---
## 10. Change in Cascade Metrics (Δ) by Wealth Decile

In [ ]:
decile_delta = df.groupby('Wealth_Decile').agg(
    Delta_Net=('Delta_Net_Cascade','mean'),
    Delta_Churn=('Delta_CFI_Churn','mean'),
    Delta_Rate=('Delta_CFI_Rate','mean'),
    IMD_Rank_Change=('IMD_Pctile_Change','mean'),
).round(2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
x = np.arange(1, 11)

for ax, col, title in [
    (axes[0,0], 'Delta_Net', 'Δ Net Cascade (2021−2011)'),
    (axes[0,1], 'Delta_Churn', 'Δ CFI Churn (2021−2011)'),
    (axes[1,0], 'Delta_Rate', 'Δ CFI Rate (2021−2011)'),
    (axes[1,1], 'IMD_Pctile_Change', 'Mean IMD Change (2010→2019)'),
]:
    vals = decile_delta[col].values
    if col == 'IMD_Pctile_Change':
        colors = ['coral' if v < 0 else 'steelblue' for v in vals]
    else:
        colors = ['coral' if v > 0 else 'steelblue' for v in vals]
    ax.bar(x, vals, color=colors, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Wealth Decile')
    ax.set_ylabel(f'Mean {title}')
    ax.set_title(title)
    ax.set_xticks(x)

plt.tight_layout()

save_path = OUTPUT_DIR / 'fig14_delta_by_decile.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

display(decile_delta)

##### Interpretation
- **Δ CFI Churn** is negative across *all* deciles (cascade churn fell everywhere from 2011→2021), consistent with the overall migration decline in the COVID-era Census.
- However, the *magnitude* of decline varies: wealthiest deciles (8–10) lost less churn than middle deciles, suggesting the migration slowdown was uneven.
- **Δ Net Cascade** shows a mixed pattern but trends towards more *negative* values in deprived deciles — cascade pressure slightly weakened in the poorest areas.
- The IMD Change panel confirms the strongest deprivation declines occurred in decile 1 (most deprived).

---
## 11. Multivariate View: CFI Churn vs Net Cascade, coloured by IMD Change

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    df['CFI_Churn_21'], df['Net_Cascade_21'],
    c=df['IMD_Pctile_Change'], cmap='RdBu', s=20, alpha=0.5,
    edgecolors='grey', linewidth=0.3, vmin=-20, vmax=10)
ax.set_xlabel('CFI Churn (2021) — Cascade Intensity')
ax.set_ylabel('Net Cascade (2021) — Cascade Direction')
ax.set_title('CFI Churn vs Net Cascade (2021)\nColoured by IMD Change (blue = became less deprived)')
ax.axhline(0, color='black', lw=0.8, ls='--')
cbar = fig.colorbar(sc)
cbar.set_label('IMD Change (2010→2019)')
plt.tight_layout()

save_path = OUTPUT_DIR / 'fig15_churn_vs_netcascade_imd.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

##### Interpretation
This plot reveals why CFI Churn and Net Cascade capture *different aspects* of the cascade:
- The **upper-right quadrant** (high churn + positive Net Cascade) contains MSOAs with both high intensity *and* net inward gentrification pressure — these are the strongest gentrification candidates.
- The **upper-left quadrant** (high churn + negative Net Cascade) contains high-turnover MSOAs where *outflow* dominates — displacement exporters.
- Blue (less deprived) colouring appears in both quadrants, confirming that deprivation decline is associated with cascade *intensity* (churn), not just direction.

---
## 12. MSOA Typology Comparison: Net Cascade vs CFI Churn

In [ ]:
# Net Cascade typology (threshold = 0)
def classify_net(row):
    h11 = row['Net_Cascade_11'] > 0
    h21 = row['Net_Cascade_21'] > 0
    if not h11 and h21: return 'Emerging'
    elif h11 and h21: return 'Sustained'
    elif h11 and not h21: return 'Stalled'
    else: return 'Stable'

# CFI Churn typology (threshold = median)
churn_med_11 = df['CFI_Churn_11'].median()
churn_med_21 = df['CFI_Churn_21'].median()
def classify_churn(row):
    h11 = row['CFI_Churn_11'] > churn_med_11
    h21 = row['CFI_Churn_21'] > churn_med_21
    if not h11 and h21: return 'Emerging'
    elif h11 and h21: return 'Sustained'
    elif h11 and not h21: return 'Stalled'
    else: return 'Stable'

df['Type_NetCascade'] = df.apply(classify_net, axis=1)
df['Type_CFIChurn'] = df.apply(classify_churn, axis=1)

print('=== Typology Counts ===')
print('\nNet Cascade:')
print(df['Type_NetCascade'].value_counts().to_string())
print('\nCFI Churn:')
print(df['Type_CFIChurn'].value_counts().to_string())
print('\nCross-tabulation:')
display(pd.crosstab(df['Type_NetCascade'], df['Type_CFIChurn'], margins=True))

In [ ]:
type_colors = {'Emerging': '#d73027', 'Sustained': '#fc8d59',
               'Stalled': '#91bfdb', 'Stable': '#4575b4'}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Net Cascade typology
ax = axes[0]
for gtype, color in type_colors.items():
    mask = df['Type_NetCascade'] == gtype
    ax.scatter(df.loc[mask, 'Net_Cascade_11'], df.loc[mask, 'Net_Cascade_21'],
               c=color, label=f'{gtype} (n={mask.sum()})', s=25, alpha=0.6,
               edgecolors='grey', linewidth=0.3)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(0, color='black', lw=0.8, ls='--')
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k:', alpha=0.3)
ax.set_xlabel('Net Cascade (2011)'); ax.set_ylabel('Net Cascade (2021)')
ax.set_title('A. Net Cascade Typology'); ax.legend(fontsize=9)

# CFI Churn typology
ax = axes[1]
for gtype, color in type_colors.items():
    mask = df['Type_CFIChurn'] == gtype
    ax.scatter(df.loc[mask, 'CFI_Churn_11'], df.loc[mask, 'CFI_Churn_21'],
               c=color, label=f'{gtype} (n={mask.sum()})', s=25, alpha=0.6,
               edgecolors='grey', linewidth=0.3)
ax.axhline(churn_med_21, color='black', lw=0.8, ls='--')
ax.axvline(churn_med_11, color='black', lw=0.8, ls='--')
ax.set_xlabel('CFI Churn (2011)'); ax.set_ylabel('CFI Churn (2021)')
ax.set_title(f'B. CFI Churn Typology (median threshold)')
ax.legend(fontsize=9)

plt.suptitle('MSOA Gentrification Typology: Two Formulations Compared', fontsize=14, y=1.01)
plt.tight_layout()

save_path = OUTPUT_DIR / 'fig16_typology_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
# Typology validation
print('=== Mean IMD Change by Typology ===\n')
for typ_col, label in [('Type_NetCascade', 'Net Cascade'), ('Type_CFIChurn', 'CFI Churn')]:
    print(f'{label} Typology:')
    summary = df.groupby(typ_col).agg(
        Count=('msoa11cd', 'count'),
        Mean_IMD_2010=('IMD_2010', 'mean'),
        Mean_IMD_Rank_Change=('IMD_Pctile_Change', 'mean'),
    ).round(2)
    display(summary)
    # ANOVA
    groups = [g['IMD_Pctile_Change'].values for _, g in df.groupby(typ_col)]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f'  One-way ANOVA: F = {f_stat:.2f}, p = {p_val:.2e}\n')

##### Comparison
- The **Net Cascade** typology produces a clearer separation of IMD Change: Sustained/Stalled MSOAs (which had positive cascade pressure in 2011) show the largest deprivation declines (−5.7 to −6.2), while Stable MSOAs show only −2.4.
- The **CFI Churn** typology also separates groups, but the Sustained category is broader (427 MSOAs vs 396) because median-based thresholds capture volume effects that include non-gentrifying high-turnover areas.
- The cross-tabulation reveals 200 MSOAs classified as "Stable" by Net Cascade but "Sustained" by CFI Churn — these are areas with high cascade *activity* but where inflow and outflow balance out (Net Cascade ≈ 0).

---
## 13. Borough-Level: CFI Churn Analysis

In [ ]:
borough = df.groupby('ladnm').agg(
    N=('msoa11cd','count'),
    Mean_CFI_Churn_11=('CFI_Churn_11','mean'),
    Mean_CFI_Churn_21=('CFI_Churn_21','mean'),
    Mean_CFI_Rate_21=('CFI_Rate_21','mean'),
    Total_Net_Cascade_21=('Net_Cascade_21','sum'),
    Mean_IMD_Rank_Change=('IMD_Pctile_Change','mean'),
).round(2)
borough['Delta_Churn'] = (borough['Mean_CFI_Churn_21'] - borough['Mean_CFI_Churn_11']).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

bs = borough.sort_values('Mean_CFI_Churn_21', ascending=True)
med = borough['Mean_CFI_Churn_21'].median()
colors = ['#d73027' if v > med else '#4575b4' for v in bs['Mean_CFI_Churn_21']]
axes[0].barh(bs.index, bs['Mean_CFI_Churn_21'], color=colors, edgecolor='white')
axes[0].set_xlabel('Mean CFI Churn (2021)')
axes[0].set_title('A. Average CFI Churn by Borough (2021)')

bs2 = borough.sort_values('Delta_Churn', ascending=True)
colors2 = ['coral' if v > 0 else 'steelblue' for v in bs2['Delta_Churn']]
axes[1].barh(bs2.index, bs2['Delta_Churn'], color=colors2, edgecolor='white')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Δ CFI Churn (2021 − 2011)')
axes[1].set_title('B. Change in CFI Churn by Borough')

plt.tight_layout()

save_path = OUTPUT_DIR / 'fig17_borough_churn.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

display(borough.sort_values('Mean_CFI_Churn_21', ascending=False))

---
## 14. Top 20 MSOAs by CFI Churn (2021)

In [ ]:
top20 = df.nlargest(20, 'CFI_Churn_21')[
    ['msoa11cd','ladnm','Wealth_Decile','CFI_Churn_21','CFI_Rate_21',
     'Net_Cascade_21','Inflow_Wealthier_21','Outflow_Poorer_21','IMD_2010','IMD_Pctile_Change']
].reset_index(drop=True)

print('=== Top 20 MSOAs by CFI Churn (2021) ===')
display(top20)

##### Observation
The highest-churn MSOAs cluster in Lambeth, Islington, Wandsworth, Tower Hamlets, and Hillingdon — boroughs with well-documented gentrification dynamics. Many have *negative* Net Cascade (outflow exceeds inflow), showing that CFI Churn captures displacement-exporting areas that Net Cascade alone would classify as "not gentrifying".

---
## 15. Summary & Recommendations

### Key Findings

1. **Net Cascade remains the strongest single predictor** of deprivation change (r ≈ −0.42). Its monotonic relationship with wealth deciles makes it the most interpretable measure of gentrification *direction*.

2. **CFI Churn adds complementary value** as a measure of cascade *intensity* (r ≈ −0.25 to −0.30). It captures the total volume of displacement activity regardless of whether the net direction is positive or negative. This is analytically useful for:
   - Identifying high-turnover transitional areas
   - The bivariate Pressure × Displacement analysis (Section 8)
   - Distinguishing "balanced churn" MSOAs where Net Cascade ≈ 0 but intense flows are occurring

3. **CFI Rate does not validate** against deprivation change (r ≈ −0.05, ns). The normalisation by total migration cancels out the volume signal, and the multiplicative interaction term peaks at mid-deciles rather than following the expected gentrification gradient. **Recommend deprioritising this metric** in the dissertation analysis.

4. **The delta metrics (Δ) are weak predictors** (r < 0.2), likely because the 2021 Census captured COVID-era migration patterns that differ from the 2010–2019 deprivation trajectory being validated against. The *level* of cascade activity at each time point is more informative than the change.

### Recommended Approach for Dissertation
- Use **Net Cascade** as the primary gentrification index (strongest validation, clearest gradient).
- Use **CFI Churn** as a secondary intensity measure for the bivariate analysis and typology refinement.
- Report **CFI Rate** in an appendix with the methodological note that normalisation eliminates the predictive signal.
- Retain the **Net Cascade typology** (Emerging/Sustained/Stalled/Stable) as the primary classification.